# LLMDet Evaluation and Visualization Template

This notebook is designed for your current situation: **you do not have `predictions.pkl` yet, but you already have `work_dirs`, checkpoints, and training logs**.

It is organized into two parts:
1. Analyze the training logs, checkpoints, and current files under `work_dirs`.
2. Provide placeholders for future `predictions.pkl` analysis, so you can continue with offline evaluation and visualization once prediction outputs are available.


## 0. Path setup

Please update the paths below to match the actual locations on your server.

In [ ]:
from pathlib import Path
import os

# ====== Update these paths based on your environment ======
PROJECT_ROOT = Path('/root/LLMDet')
WORK_DIR = PROJECT_ROOT / 'work_dirs' / 'grounding_dino_t_for_alignment'
CONFIG_SNAPSHOT = WORK_DIR / 'grounding_dino_t_for_alignment.py'
LAST_CKPT_FILE = WORK_DIR / 'last_checkpoint'
PRED_PKL = WORK_DIR / 'predictions.pkl'   # This file may not exist yet

print('PROJECT_ROOT =', PROJECT_ROOT)
print('WORK_DIR =', WORK_DIR)
print('WORK_DIR exists?', WORK_DIR.exists())
print('CONFIG_SNAPSHOT exists?', CONFIG_SNAPSHOT.exists())
print('LAST_CKPT_FILE exists?', LAST_CKPT_FILE.exists())
print('PRED_PKL exists?', PRED_PKL.exists())


## 1. Inspect the current files in `work_dir`

This step helps you check what files are already available.

In [ ]:
if WORK_DIR.exists():
    for p in sorted(WORK_DIR.iterdir()):
        kind = 'DIR ' if p.is_dir() else 'FILE'
        size_mb = p.stat().st_size / 1024 / 1024 if p.is_file() else None
        if size_mb is None:
            print(f'[{kind}] {p.name}')
        else:
            print(f'[{kind}] {p.name:40s}  {size_mb:8.2f} MB')
else:
    print('WORK_DIR does not exist. Please check the path settings above.')


## 2. Read `last_checkpoint`

This file usually tells you which checkpoint is considered the latest one.

In [ ]:
if LAST_CKPT_FILE.exists():
    text = LAST_CKPT_FILE.read_text().strip()
    print('Contents of last_checkpoint:')
    print(text)
else:
    print('No last_checkpoint file found.')


## 3. Find all checkpoint files

This helps you confirm how many checkpoints are currently available.

In [ ]:
ckpts = sorted(WORK_DIR.glob('*.pth')) if WORK_DIR.exists() else []
print(f'Found {len(ckpts)} checkpoint file(s):')
for p in ckpts:
    print(f'- {p.name:30s}  {p.stat().st_size / 1024 / 1024:.2f} MB')


## 4. Search for log files

MMEngine commonly stores `.log` and `.json` files. This cell searches for possible training records.

In [ ]:
log_files = []
if WORK_DIR.exists():
    for ext in ['*.log', '*.json', '*.txt']:
        log_files.extend(WORK_DIR.rglob(ext))
log_files = sorted(set(log_files))

print(f'Found {len(log_files)} possible log/record file(s):')
for p in log_files:
    try:
        size_mb = p.stat().st_size / 1024 / 1024
        print(f'- {p.relative_to(WORK_DIR)}  ({size_mb:.2f} MB)')
    except Exception:
        print('-', p)


## 5. Preview log file contents

This prints the first few lines of each log file so you can quickly identify the main training log.

In [ ]:
def preview_text_file(path, n=10):
    print('=' * 80)
    print(path)
    print('-' * 80)
    try:
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            for i, line in enumerate(f):
                if i >= n:
                    break
                print(line.rstrip())
    except Exception as e:
        print('Failed to read file:', e)

for p in log_files[:8]:
    preview_text_file(p, n=8)


## 6. Extract training metrics from MMEngine text logs

This step is designed for MMEngine log lines such as:

`Iter(train) [30100/150000] ... loss: 6.9907 ... loss_cls: 0.2133 ...`

It automatically extracts:
- iteration index
- learning rate
- total loss
- `loss_cls` / `loss_bbox` / `loss_iou`
- `grad_norm`


In [ ]:
import re
import pandas as pd

pattern_iter = re.compile(r'Iter\(train\) \[\s*(\d+)/(\d+)\]')
pattern_kv = re.compile(r'([A-Za-z0-9_\.]+):\s*([0-9eE\+\-\.]+)')

def parse_mmengine_log_text(log_path):
    rows = []
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if 'Iter(train)' not in line:
                continue
            m = pattern_iter.search(line)
            if not m:
                continue
            row = {
                'iter': int(m.group(1)),
                'max_iter': int(m.group(2)),
            }
            for k, v in pattern_kv.findall(line):
                try:
                    row[k] = float(v)
                except Exception:
                    pass
            rows.append(row)
    return pd.DataFrame(rows)

# Try to parse all text-based logs and keep the first valid one
candidate_logs = [p for p in log_files if p.suffix in {'.log', '.txt'}]
parsed = None
parsed_from = None

for p in candidate_logs:
    try:
        df_try = parse_mmengine_log_text(p)
        if len(df_try) > 0:
            parsed = df_try
            parsed_from = p
            break
    except Exception:
        pass

if parsed is not None:
    print('Successfully parsed training log from:', parsed_from)
    print(parsed.head())
    print('rows =', len(parsed))
else:
    print('No parsable MMEngine training records were found.')


## 7. Visualize training curves

If the previous step succeeds, this section plots:
- total loss
- `loss_cls` / `loss_bbox` / `loss_iou`
- `grad_norm`
- learning rate


In [ ]:
import matplotlib.pyplot as plt

if parsed is not None and len(parsed) > 0:
    cols_to_plot = ['loss', 'loss_cls', 'loss_bbox', 'loss_iou', 'grad_norm', 'lr']
    available = [c for c in cols_to_plot if c in parsed.columns]
    print('Available columns for plotting:', available)
    for col in available:
        plt.figure(figsize=(8, 4))
        plt.plot(parsed['iter'], parsed[col])
        plt.xlabel('Iteration')
        plt.ylabel(col)
        plt.title(f'{col} vs Iteration')
        plt.grid(True, alpha=0.3)
        plt.show()
else:
    print('No parsed results are available for plotting.')


## 8. Training summary table

This section provides a concise summary of the training statistics extracted from the log.

In [ ]:
if parsed is not None and len(parsed) > 0:
    summary = {}
    summary['n_records'] = len(parsed)
    summary['first_iter'] = int(parsed['iter'].min())
    summary['last_iter'] = int(parsed['iter'].max())
    for col in ['loss', 'loss_cls', 'loss_bbox', 'loss_iou', 'grad_norm', 'lr']:
        if col in parsed.columns:
            summary[f'{col}_first'] = float(parsed[col].iloc[0])
            summary[f'{col}_last'] = float(parsed[col].iloc[-1])
            summary[f'{col}_min'] = float(parsed[col].min())
            summary[f'{col}_max'] = float(parsed[col].max())
    summary_df = pd.DataFrame([summary]).T.reset_index()
    summary_df.columns = ['metric', 'value']
    display(summary_df)
else:
    print('No training summary is available.')


## 9. Placeholder: load `predictions.pkl`

Once you run test with `--out .../predictions.pkl`, you can directly use the cell below.

Example command:

```bash
cd /root/LLMDet
bash dist_test.sh configs/grounding_dino_t_for_alignment.py \
  work_dirs/grounding_dino_t_for_alignment/iter_30000.pth 8 \
  --out work_dirs/grounding_dino_t_for_alignment/predictions.pkl
```


In [ ]:
import pickle

if PRED_PKL.exists():
    with open(PRED_PKL, 'rb') as f:
        preds = pickle.load(f)
    print('type(preds) =', type(preds))
    try:
        print('len(preds) =', len(preds))
    except Exception:
        pass
    try:
        print('type(preds[0]) =', type(preds[0]))
        print('preds[0] =')
        print(preds[0])
    except Exception as e:
        print('Unable to inspect preds[0]:', e)
else:
    print('`predictions.pkl` is not available yet. Run test first and export it using `--out`.')


## 10. Placeholder: inspect prediction item structure

This section is for future use after `predictions.pkl` becomes available. Since different projects may dump slightly different structures, this cell helps you inspect the format first.

In [ ]:
def inspect_prediction_item(item):
    print('item type =', type(item))
    if isinstance(item, dict):
        print('keys =', list(item.keys()))
        for k, v in item.items():
            print(f'  {k}: {type(v)}')
    else:
        attrs = [a for a in dir(item) if not a.startswith('_')]
        print('attrs (first 30) =', attrs[:30])

if PRED_PKL.exists():
    inspect_prediction_item(preds[0])
else:
    print('`predictions.pkl` is not available yet.')


## 11. Suggested next steps

Once you have `predictions.pkl`, the next modules to add are:

1. A parser that converts `predictions.pkl` into a DataFrame  
2. A matching function between predictions and GT annotations  
3. TP / FP / FN statistics  
4. Score distribution plots  
5. Class distribution plots  
6. Error case visualization  

At that point, you only need to inspect the structure of `predictions.pkl`, and this notebook can be extended into a full offline evaluation notebook.
